# 03 — Eckende quantile model

Build completed-order data, train Q50/Q80/Q85/Q90 models and compare current versus simulated ON_TIME performance.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

In [ ]:
import json
import pandas as pd

from ml_wartungsplan.data.build_deadline_dataset import build_deadline_dataset
from ml_wartungsplan.models.deadline import train_deadline_models
from ml_wartungsplan.settings import load_settings, resolve_project_path

settings = load_settings()
paths = settings["paths"]
deadline_settings = settings["deadline_model"]

In [ ]:
deadline_dataset = build_deadline_dataset(
    excel_path=resolve_project_path(paths["raw_excel"]),
    output_path=resolve_project_path(paths["deadline_dataset"]),
    report_path=resolve_project_path(paths["reports_dir"]) / "deadline/data_quality.json",
    german_state=settings["project"]["german_state"],
    earliest_valid_date=deadline_settings["earliest_valid_date"],
    latest_completion_offset_days=deadline_settings["latest_completion_offset_days"],
    target_clip_workdays=deadline_settings["target_clip_workdays"],
)

print("Completed orders available:", len(deadline_dataset))
display(deadline_dataset.head())
display(deadline_dataset["actual_extension_workdays_raw"].describe(
    percentiles=[0.5, 0.75, 0.8, 0.85, 0.9, 0.95]
))

In [ ]:
metrics = train_deadline_models(
    dataset_path=resolve_project_path(paths["deadline_dataset"]),
    model_path=resolve_project_path(paths["deadline_model"]),
    report_dir=resolve_project_path(paths["reports_dir"]) / "deadline",
    quantiles=[float(q) for q in deadline_settings["quantiles"]],
    selected_quantile=float(deadline_settings["selected_quantile"]),
    test_fraction=float(deadline_settings["test_fraction"]),
    random_state=settings["project"]["random_state"],
    text_svd_components=int(deadline_settings["text_svd_components"]),
    max_text_features=int(deadline_settings["max_text_features"]),
    min_document_frequency=int(deadline_settings["min_document_frequency"]),
    max_training_rows=deadline_settings.get("max_training_rows"),
)

display(pd.DataFrame(metrics["quantiles"]).T)
display(pd.Series(metrics["business_comparison"]).to_frame("value"))

In [ ]:
test_results = pd.read_csv(
    PROJECT_ROOT / "reports/deadline/test_predictions.csv"
)

display(test_results.head())
display(
    test_results[
        test_results["would_be_on_time_q85"] == 0
    ].sort_values("predicted_extension_q85", ascending=False).head(30)
)

Do not choose Q85 merely because it is the default. Compare Q80, Q85 and Q90 with planners using:

- achieved ON_TIME coverage
- average extension
- excessive padding
- remaining late orders
- work-centre-specific performance